# Return and stationarity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
close = pd.read_csv("../data/raw/close.csv", parse_dates=["Date"], index_col="Date")
log_price = np.log(close)
returns = log_price.diff().dropna()
returns.head()

In [ ]:
# ACF (left) and PACF (right) of daily log returns per security
securities = returns.columns
lags = 40

fig, axes = plt.subplots(len(securities), 2, figsize=(11, 3 * len(securities)))

for row, sec in enumerate(securities):
    plot_acf(returns[sec], ax=axes[row, 0], lags=lags, title=f"{sec} - ACF")
    plot_pacf(returns[sec], ax=axes[row, 1], lags=lags, method="ywm", title=f"{sec} - PACF")

fig.tight_layout()

## ACF / PACF of price levels

In [ ]:
# ACF (left) and PACF (right) of raw close prices per security
fig, axes = plt.subplots(len(securities), 2, figsize=(11, 3 * len(securities)))

for row, sec in enumerate(securities):
    plot_acf(close[sec], ax=axes[row, 0], lags=lags, title=f"{sec} - ACF (price)")
    plot_pacf(close[sec], ax=axes[row, 1], lags=lags, method="ywm", title=f"{sec} - PACF (price)")

fig.tight_layout()

## Stationarity: ADF test on price vs returns

In [ ]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series):
    stat, pvalue, *_ = adfuller(series, autolag="AIC")
    return stat, pvalue

In [ ]:
# ADF test on raw price level vs. log returns, per security
rows = []
for sec in close.columns:
    price_stat, price_p = run_adf(close[sec])
    ret_stat, ret_p = run_adf(returns[sec])
    rows.append({
        "security": sec,
        "price ADF stat": price_stat,
        "price p-value": price_p,
        "returns ADF stat": ret_stat,
        "returns p-value": ret_p,
    })

adf_summary = pd.DataFrame(rows).set_index("security")
adf_summary